In [ ]:
pip install fuzzywuzzy

In [ ]:
pip install spacy pandas rapidfuzz python-Levenshtein tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.4 MB/s eta 0:00:00


In [3]:
pip install fuzzywuzzy

In [4]:
!python -m spacy download fr_core_news_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 78.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
from fuzzywuzzy import fuzz
from difflib import SequenceMatcher
import pandas as pd

# Charger le modèle spaCy (français)
nlp = spacy.load("fr_core_news_sm")

# Charger les fichiers CSV
df_adresses = pd.read_csv("Adresse1.csv")  # Fichier contenant les adresses complètes
df_villes = pd.read_csv("villes.csv")  # Liste officielle des villes

# Liste des villes
catalogue_villes = df_villes['NomVille'].dropna().str.upper().tolist()  # On ignore les valeurs NaN

# Fonction pour appliquer l'expansion des abréviations
def expand_abbreviations(adresse):
    # Par exemple, remplace MY par Moulay Yacoub
    adresse = adresse.replace("MY", "MOULAY YACOUB")
    return adresse

# Fonction pour extraire la ville d'une adresse
def extraire_ville(adresse, villes):
    adresse_expanded = expand_abbreviations(adresse.upper())  # Expansion des abréviations et mise en majuscules
    best_match = None
    best_score = 0

    # Recherche de la ville exacte
    for ville in villes:
        if ville in adresse_expanded:  # Si une correspondance exacte est trouvée
            return ville.title()

    # Calculer la similarité avec fuzzywuzzy ou SequenceMatcher
    for ville in villes:
        score = fuzz.partial_ratio(adresse_expanded, ville.upper())  # Mesure de similarité
        if score > best_score:
            best_match = ville
            best_score = score

    # Retourner la ville si la similarité est suffisante
    if best_score > 80:  # Seulement si le score de similarité est suffisant
        return best_match.title()

    return None  # Si aucune ville n'est trouvée

# Appliquer la fonction sur toutes les adresses
df_adresses['Ville'] = df_adresses['AdresseComplete'].apply(lambda x: extraire_ville(x, catalogue_villes))

# Sauvegarder le fichier final
df_adresses[['AdresseComplete', 'Ville']].to_csv("adresses_nettoyees.csv", index=False)

In [ ]:
import spacy
from difflib import get_close_matches
import pandas as pd
import re

# Charger le modèle spaCy français
nlp = spacy.load("fr_core_news_sm")

# Charger les fichiers CSV
df_adresses = pd.read_csv("Adresse1.csv")  # Contient les adresses complètes
df_villes = pd.read_csv("villes.csv")  # Contient la liste des villes

# Nettoyer le dataset des villes (enlever les NaN et mettre en majuscules pour homogénéiser)
df_villes = df_villes.dropna(subset=['NomVille'])  # Supprimer les lignes où NomVille est NaN
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()  # Liste des villes en majuscules

# Vérifier si les villes sont correctement chargées
print("Liste des villes du catalogue :")
print(catalogue_villes[:10])  # Affiche les 20 premières villes pour vérification

# Fonction pour extraire une ville similaire avec un seuil de similarité
def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None

    # Tokenisation et normalisation avec spaCy
    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])

    # Nettoyer les espaces et caractères spéciaux
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    print(f"Adresse normalisée : {normalized_input}")  # Débogage : affiche l'adresse normalisée

    # 1. Vérification directe de la correspondance exacte (avant get_close_matches)
    for ville in villes:
        if ville in normalized_input.upper():
            print(f"Correspondance exacte trouvée : {ville}")  # Débogage : correspondance exacte
            return ville.title()

    # 2. Chercher les villes similaires dans le catalogue en fonction de la similarité
    matches = get_close_matches(normalized_input, villes, n=1, cutoff=seuil_similarite)

    # Débogage : vérifier si des correspondances sont trouvées
    if matches:
        print(f"Correspondance trouvée via similarité : {matches[0]}")  # Débogage : ville similaire trouvée
        return matches[0].title()
    else:
        print("Aucune correspondance trouvée.")  # Débogage : si aucune correspondance n'est trouvée

    return None  # Retourne None si aucune ville similaire trouvée

# Appliquer la fonction pour extraire la ville similaire
df_adresses['Ville'] = df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes))

# Vérification de l'extraction des villes
print("\nAdresses avec ville extraite :")
print(df_adresses[['AdresseComplete', 'Ville']].head(20))  # Affiche les 20 premières lignes

# Fonction pour retirer la ville de l'adresse et obtenir uniquement l'adresse restante
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)

# Séparer les adresses avec et sans ville
df_avec_ville = df_adresses.dropna(subset=['Ville'])  # Garde celles avec une ville
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville'])  # Garde les adresses sans ville

# Sauvegarder le fichier final
df_adresses[['Adresse', 'Ville']].to_csv("adresses_nettoyees.csv", index=False)

# Affichage des résultats
print("\n📍 Adresses avec ville détectée :")
print(df_avec_ville[['Adresse', 'Ville']], "\n")

print("⚠️ Adresses sans ville détectée :")
print(df_sans_ville)

Liste des villes du catalogue :
['ABADOU', 'ABETEH', 'AFFELA IRHIR', 'AFOURER', 'AGADIR', 'AGDEZ', 'AGHBALA', 'AGHBALOU NKERDOUSS', 'AGHWINIT', 'AGOURAI']
Adresse normalisée : DR EL FIDE MEKKES MOULAY YACOUB
Correspondance exacte trouvée : MOULAY YACOUB
Adresse normalisée : COOP MASTOURA SBAA ROUADI MY YACOUB
Aucune correspondance trouvée.
Adresse normalisée : 4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN FES
Correspondance exacte trouvée : FES
Adresse normalisée : DOUAR OLD BEN ABDELLAH OUDAYA FES BLIE
Correspondance exacte trouvée : FES
Adresse normalisée : AVE ALLAL BEN ABDELLAH 7 RUE AGHBALOU N
Aucune correspondance trouvée.
Adresse normalisée : Nø 58 AV IBN EL KHATIB / FES AGHBALOU N
Correspondance exacte trouvée : FES
Adresse normalisée : 40 RUE TOULOUSE CIL CASA 2
Aucune correspondance trouvée.

Adresses avec ville extraite :
                                     AdresseComplete          Ville
0                   DR EL FIDE MEKKES MOULAY YACOUB   Moulay Yacoub
1               

In [ ]:
import spacy
from fuzzywuzzy import process  # Utilisation de fuzzywuzzy pour un matching plus flexible
import pandas as pd
import re

# Charger le modèle spaCy français
nlp = spacy.load("fr_core_news_sm")

# Charger les fichiers CSV
df_adresses = pd.read_csv("Adresse1.csv")  # Contient les adresses complètes
df_villes = pd.read_csv("villes.csv")  # Contient la liste des villes

# Nettoyer le dataset des villes (enlever les NaN et mettre en majuscules pour homogénéiser)
df_villes = df_villes.dropna(subset=['NomVille'])  # Supprimer les lignes où NomVille est NaN
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()  # Liste des villes en majuscules

# Vérifier si les villes sont correctement chargées
print("Liste des villes du catalogue :")
print(catalogue_villes[:10])  # Affiche les 10 premières villes pour vérification

# Fonction pour extraire une ville similaire avec un seuil de similarité
def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None

    # Tokenisation et normalisation avec spaCy
    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])

    # Nettoyer les espaces et caractères spéciaux
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    print(f"Adresse normalisée : {normalized_input}")  # Débogage : affiche l'adresse normalisée

    # 1. Vérification directe de la correspondance exacte (avant get_close_matches)
    for ville in villes:
        if ville in normalized_input.upper():
            print(f"Correspondance exacte trouvée : {ville}")  # Débogage : correspondance exacte
            return ville.title()

    # 2. Chercher les villes similaires dans le catalogue en fonction de la similarité
    matches = process.extractOne(normalized_input.upper(), villes)

    # Débogage : vérifier si des correspondances sont trouvées
    if matches:
        best_match, score = matches
        if score >= seuil_similarite * 100:  # Seuil ajusté en pourcentage
            print(f"Correspondance trouvée via similarité : {best_match}")  # Débogage : ville similaire trouvée
            return best_match.title()
        else:
            print("Aucune correspondance suffisante trouvée.")  # Débogage : si la correspondance n'est pas assez proche
    return None  # Retourne None si aucune ville similaire trouvée

# Fonction pour compléter les noms de villes incomplets
def completer_ville_incomplete(ville_incomplete, villes, seuil_similarite=100):
    """Complète le nom de la ville si un préfixe est fourni mais incomplet."""
    if ville_incomplete:
        # Utilisation de fuzzywuzzy pour trouver la meilleure correspondance
        best_match, score = process.extractOne(ville_incomplete.upper(), villes)
        if score >= seuil_similarite:
            return best_match.title()  # Retourne la ville complète trouvée
    return ville_incomplete  # Si aucune correspondance avec un score élevé, retourne le nom incomplet

# Appliquer la fonction pour extraire la ville similaire
df_adresses['Ville'] = df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes))

# Compléter les villes incomplètes en parcourant les adresses
df_adresses['Ville'] = df_adresses['Ville'].apply(lambda ville: completer_ville_incomplete(ville, catalogue_villes) if ville else None)

# Vérification de l'extraction des villes
print("\nAdresses avec ville extraite :")
print(df_adresses[['AdresseComplete', 'Ville']].head(20))  # Affiche les 20 premières lignes

# Fonction pour retirer la ville de l'adresse et obtenir uniquement l'adresse restante
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)

# Séparer les adresses avec et sans ville
df_avec_ville = df_adresses.dropna(subset=['Ville'])  # Garde celles avec une ville
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville'])  # Garde les adresses sans ville

# Sauvegarder le fichier final
df_adresses[['Adresse', 'Ville']].to_csv("adresses_nettoyees.csv", index=False)

# Affichage des résultats
print("\n📍 Adresses avec ville détectée :")
print(df_avec_ville[['Adresse', 'Ville']], "\n")

print("⚠️ Adresses sans ville détectée :")
print(df_sans_ville)


Liste des villes du catalogue :
['ABADOU', 'ABETEH', 'AFFELA IRHIR', 'AFOURER', 'AGADIR', 'AGDEZ', 'AGHBALA', 'AGHBALOU NKERDOUSS', 'AGHWINIT', 'AGOURAI']
Adresse normalisée : DR EL FIDE MEKKES MOULAY YACOUB
Correspondance exacte trouvée : MOULAY YACOUB
Adresse normalisée : COOP MASTOURA SBAA ROUADI MY YACOUB
Correspondance trouvée via similarité : MOULAY YACOUB
Adresse normalisée : 4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN FES
Correspondance exacte trouvée : FES
Adresse normalisée : DOUAR OLD BEN ABDELLAH OUDAYA FES BLIE
Correspondance exacte trouvée : FES
Adresse normalisée : AVE ALLAL BEN ABDELLAH 7 RUE AGHBALOU N
Correspondance trouvée via similarité : AGHBALOU NKERDOUSS
Adresse normalisée : Nø 58 AV IBN EL KHATIB / FES AGHBALOU N
Correspondance exacte trouvée : FES
Adresse normalisée : 40 RUE TOULOUSE CIL CASA 2
Correspondance trouvée via similarité : ASSA

Adresses avec ville extraite :
                                     AdresseComplete               Ville
0             

In [ ]:
import spacy
from fuzzywuzzy import process  # Utilisation de fuzzywuzzy pour un matching plus flexible
import pandas as pd
import re

# Charger le modèle spaCy français
nlp = spacy.load("fr_core_news_sm")

# Charger les fichiers CSV
df_adresses = pd.read_csv("Adresse1.csv")  # Contient les adresses complètes
df_villes = pd.read_csv("villes.csv")  # Contient la liste des villes

# Nettoyer le dataset des villes (enlever les NaN et mettre en majuscules pour homogénéiser)
df_villes = df_villes.dropna(subset=['NomVille'])  # Supprimer les lignes où NomVille est NaN
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()  # Liste des villes en majuscules

# Vérifier si les villes sont correctement chargées
print("Liste des villes du catalogue :")
print(catalogue_villes[:10])  # Affiche les 10 premières villes pour vérification

# Fonction pour extraire une ville similaire avec un seuil de similarité
def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None

    # Tokenisation et normalisation avec spaCy
    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])

    # Nettoyer les espaces et caractères spéciaux
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    print(f"Adresse normalisée : {normalized_input}")  # Débogage : affiche l'adresse normalisée

    # 1. Vérification directe de la correspondance exacte (avant get_close_matches)
    for ville in villes:
        if ville in normalized_input.upper():
            print(f"Correspondance exacte trouvée : {ville}")  # Débogage : correspondance exacte
            return ville.title()

    # 2. Chercher les villes similaires dans le catalogue en fonction de la similarité
    matches = process.extractOne(normalized_input.upper(), villes)

    # Débogage : vérifier si des correspondances sont trouvées
    if matches:
        best_match, score = matches
        if score >= seuil_similarite * 100:  # Seuil ajusté en pourcentage
            print(f"Correspondance trouvée via similarité : {best_match}")  # Débogage : ville similaire trouvée
            return best_match.title()
        else:
            print(f"Aucune correspondance suffisante trouvée, score : {score}")  # Débogage : si la correspondance n'est pas assez proche
    return None  # Retourne None si aucune ville similaire trouvée

# Fonction pour compléter les noms de villes incomplets
def completer_ville_incomplete(ville_incomplete, villes, seuil_similarite=80):
    """Complète le nom de la ville si un préfixe est fourni mais incomplet."""
    if ville_incomplete:
        # Utilisation de fuzzywuzzy pour trouver la meilleure correspondance
        best_match, score = process.extractOne(ville_incomplete.upper(), villes)
        if score >= seuil_similarite:  # Si le score est supérieur ou égal à 80 %
            return best_match.title()  # Retourne la ville complète trouvée
        else:
            print(f"Ville incomplète ignorée : {ville_incomplete}, score : {score}")  # Débogage : ville ignorée si le score est trop bas
    return None  # Retourne None si aucune correspondance n'est trouvée ou si le score est trop faible

# Appliquer la fonction pour extraire la ville similaire
df_adresses['Ville'] = df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes))

# Compléter les villes incomplètes en parcourant les adresses
df_adresses['Ville'] = df_adresses['Ville'].apply(lambda ville: completer_ville_incomplete(ville, catalogue_villes) if ville else None)

# Vérification de l'extraction des villes
print("\nAdresses avec ville extraite :")
print(df_adresses[['AdresseComplete', 'Ville']].head(20))  # Affiche les 20 premières lignes

# Fonction pour retirer la ville de l'adresse et obtenir uniquement l'adresse restante
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)

# Séparer les adresses avec et sans ville
df_avec_ville = df_adresses.dropna(subset=['Ville'])  # Garde celles avec une ville
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville'])  # Garde les adresses sans ville

# Sauvegarder le fichier final
df_adresses[['Adresse', 'Ville']].to_csv("adresses_nettoyees.csv", index=False)

# Affichage des résultats
print("\n📍 Adresses avec ville détectée :")
print(df_avec_ville[['Adresse', 'Ville']], "\n")

print("⚠️ Adresses sans ville détectée :")
print(df_sans_ville)


Liste des villes du catalogue :
['ABADOU', 'ABETEH', 'AFFELA IRHIR', 'AFOURER', 'AGADIR', 'AGDEZ', 'AGHBALA', 'AGHBALOU NKERDOUSS', 'AGHWINIT', 'AGOURAI']
Adresse normalisée : DR EL FIDE MEKKES MOULAY YACOUB
Correspondance exacte trouvée : MOULAY YACOUB
Adresse normalisée : COOP MASTOURA SBAA ROUADI MY YACOUB
Correspondance trouvée via similarité : MOULAY YACOUB
Adresse normalisée : 4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN FES
Correspondance exacte trouvée : FES
Adresse normalisée : DOUAR OLD BEN ABDELLAH OUDAYA FES BLIE
Correspondance exacte trouvée : FES
Adresse normalisée : AVE ALLAL BEN ABDELLAH 7 RUE AGHBALOU N
Correspondance trouvée via similarité : AGHBALOU NKERDOUSS
Adresse normalisée : Nø 58 AV IBN EL KHATIB / FES AGHBALOU N
Correspondance exacte trouvée : FES
Adresse normalisée : 40 RUE TOULOUSE CIL CASA 2
Correspondance trouvée via similarité : ASSA

Adresses avec ville extraite :
                                     AdresseComplete               Ville
0             

In [ ]:
import spacy
from fuzzywuzzy import process  # Utilisation de fuzzywuzzy pour un matching plus flexible
import pandas as pd
import re

# Charger le modèle spaCy français
nlp = spacy.load("fr_core_news_sm")

# Charger les fichiers CSV
df_adresses = pd.read_csv("Adresse1.csv")  # Contient les adresses complètes
df_villes = pd.read_csv("villes.csv")  # Contient la liste des villes

# Nettoyer le dataset des villes (enlever les NaN et mettre en majuscules pour homogénéiser)
df_villes = df_villes.dropna(subset=['NomVille'])  # Supprimer les lignes où NomVille est NaN
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()  # Liste des villes en majuscules

# Liste de mots-clés pour les villes
mots_cles_villes = ['CASA', 'RABAT', 'MAROC', 'TANGER', 'MARRAKECH', 'FES']

# Vérifier si les villes sont correctement chargées
print("Liste des villes du catalogue :")
print(catalogue_villes[:10])  # Affiche les 10 premières villes pour vérification

# Fonction pour extraire une ville similaire avec un seuil de similarité
def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None

    # Tokenisation et normalisation avec spaCy
    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])

    # Nettoyer les espaces et caractères spéciaux
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    print(f"Adresse normalisée : {normalized_input}")  # Débogage : affiche l'adresse normalisée

    # Vérification des mots-clés dans l'adresse
    for mot_cle in mots_cles_villes:
        if mot_cle in normalized_input.upper():
            print(f"Mot-clé trouvé dans l'adresse : {mot_cle}")  # Débogage : mot-clé trouvé
            # Cherche la ville qui correspond au mot-clé
            for ville in villes:
                if mot_cle in ville:
                    print(f"Correspondance par mot-clé : {ville}")
                    return ville.title()

    # 1. Vérification directe de la correspondance exacte
    for ville in villes:
        if ville in normalized_input.upper():
            print(f"Correspondance exacte trouvée : {ville}")  # Débogage : correspondance exacte
            return ville.title()

    # 2. Chercher les villes similaires dans le catalogue en fonction de la similarité
    matches = process.extractOne(normalized_input.upper(), villes)

    if matches:
        best_match, score = matches
        if score >= seuil_similarite * 100:  # Seuil ajusté en pourcentage
            print(f"Correspondance trouvée via similarité : {best_match}")  # Débogage : ville similaire trouvée
            return best_match.title()
        else:
            print(f"Aucune correspondance suffisante trouvée, score : {score}")  # Débogage : si la correspondance n'est pas assez proche
    return None  # Retourne None si aucune ville similaire trouvée

# Fonction pour compléter les noms de villes incomplets
def completer_ville_incomplete(ville_incomplete, villes, seuil_similarite=80):
    """Complète le nom de la ville si un préfixe est fourni mais incomplet."""
    if ville_incomplete:
        # Utilisation de fuzzywuzzy pour trouver la meilleure correspondance
        best_match, score = process.extractOne(ville_incomplete.upper(), villes)
        if score >= seuil_similarite:  # Si le score est supérieur ou égal à 80 %
            return best_match.title()  # Retourne la ville complète trouvée
        else:
            print(f"Ville incomplète ignorée : {ville_incomplete}, score : {score}")  # Débogage : ville ignorée si le score est trop bas
    return None  # Retourne None si aucune correspondance n'est trouvée ou si le score est trop faible

# Appliquer la fonction pour extraire la ville similaire
df_adresses['Ville'] = df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes))

# Compléter les villes incomplètes en parcourant les adresses
df_adresses['Ville'] = df_adresses['Ville'].apply(lambda ville: completer_ville_incomplete(ville, catalogue_villes) if ville else None)

# Vérification de l'extraction des villes
print("\nAdresses avec ville extraite :")
print(df_adresses[['AdresseComplete', 'Ville']].head(20))  # Affiche les 20 premières lignes

# Fonction pour retirer la ville de l'adresse et obtenir uniquement l'adresse restante
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)

# Séparer les adresses avec et sans ville
df_avec_ville = df_adresses.dropna(subset=['Ville'])  # Garde celles avec une ville
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville'])  # Garde les adresses sans ville

# Sauvegarder le fichier final
df_adresses[['Adresse', 'Ville']].to_csv("adresses_nettoyees.csv", index=False)

# Affichage des résultats
print("\n📍 Adresses avec ville détectée :")
print(df_avec_ville[['Adresse', 'Ville']], "\n")

print("⚠️ Adresses sans ville détectée :")
print(df_sans_ville)


Liste des villes du catalogue :
['ABADOU', 'ABETEH', 'AFFELA IRHIR', 'AFOURER', 'AGADIR', 'AGDEZ', 'AGHBALA', 'AGHBALOU NKERDOUSS', 'AGHWINIT', 'AGOURAI']
Adresse normalisée : DR EL FIDE MEKKES MOULAY YACOUB
Correspondance exacte trouvée : MOULAY YACOUB
Adresse normalisée : COOP MASTOURA SBAA ROUADI MY YACOUB
Correspondance trouvée via similarité : MOULAY YACOUB
Adresse normalisée : 4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN FES
Mot-clé trouvé dans l'adresse : FES
Correspondance par mot-clé : FES
Adresse normalisée : DOUAR OLD BEN ABDELLAH OUDAYA FES BLIE
Mot-clé trouvé dans l'adresse : FES
Correspondance par mot-clé : FES
Adresse normalisée : AVE ALLAL BEN ABDELLAH 7 RUE AGHBALOU N
Correspondance trouvée via similarité : AGHBALOU NKERDOUSS
Adresse normalisée : Nø 58 AV IBN EL KHATIB / FES AGHBALOU N
Mot-clé trouvé dans l'adresse : FES
Correspondance par mot-clé : FES
Adresse normalisée : 40 RUE TOULOUSE CIL CASA 2
Mot-clé trouvé dans l'adresse : CASA
Correspondance par mot-clé :

In [ ]:
import spacy
from fuzzywuzzy import process  # Utilisation de fuzzywuzzy pour un matching plus flexible
import pandas as pd
import re

# Charger le modèle spaCy français
nlp = spacy.load("fr_core_news_sm")

# Charger les fichiers CSV
df_adresses = pd.read_csv("Adresse1.csv")  # Contient les adresses complètes
df_villes = pd.read_csv("villes.csv")  # Contient la liste des villes

# Nettoyer le dataset des villes (enlever les NaN et mettre en majuscules pour homogénéiser)
df_villes = df_villes.dropna(subset=['NomVille'])  # Supprimer les lignes où NomVille est NaN
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()  # Liste des villes en majuscules

# Vérifier si les villes sont correctement chargées
print("Liste des villes du catalogue :")
print(catalogue_villes[:10])  # Affiche les 10 premières villes pour vérification

# Fonction pour extraire une ville similaire avec un seuil de similarité
def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None

    # Tokenisation et normalisation avec spaCy
    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])

    # Nettoyer les espaces et caractères spéciaux
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    print(f"Adresse normalisée : {normalized_input}")  # Débogage : affiche l'adresse normalisée

    # 1. Vérification directe de la correspondance exacte (avant get_close_matches)
    for ville in villes:
        if ville in normalized_input.upper():
            print(f"Correspondance exacte trouvée : {ville}")  # Débogage : correspondance exacte
            return ville.title()

    # 2. Chercher les villes similaires dans le catalogue en fonction de la similarité
    matches = process.extractOne(normalized_input.upper(), villes)

    # Débogage : vérifier si des correspondances sont trouvées
    if matches:
        best_match, score = matches
        if score >= seuil_similarite * 100:  # Seuil ajusté en pourcentage
            print(f"Correspondance trouvée via similarité : {best_match}")  # Débogage : ville similaire trouvée
            return best_match.title()
        else:
            print(f"Aucune correspondance suffisante trouvée, score : {score}")  # Débogage : si la correspondance n'est pas assez proche
    return None  # Retourne None si aucune ville similaire trouvée

# Fonction pour compléter les noms de villes incomplets
def completer_ville_incomplete(ville_incomplete, villes, seuil_similarite=80):
    """Complète le nom de la ville si un préfixe est fourni mais incomplet."""
    if ville_incomplete:
        # Utilisation de fuzzywuzzy pour trouver la meilleure correspondance
        best_match, score = process.extractOne(ville_incomplete.upper(), villes)
        if score >= seuil_similarite:  # Si le score est supérieur ou égal à 80 %
            return best_match.title()  # Retourne la ville complète trouvée
        else:
            print(f"Ville incomplète ignorée : {ville_incomplete}, score : {score}")  # Débogage : ville ignorée si le score est trop bas
    return None  # Retourne None si aucune correspondance n'est trouvée ou si le score est trop faible

# Appliquer la fonction pour extraire la ville similaire
df_adresses['Ville'] = df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes))

# Compléter les villes incomplètes en parcourant les adresses
df_adresses['Ville'] = df_adresses['Ville'].apply(lambda ville: completer_ville_incomplete(ville, catalogue_villes) if ville else None)

# Vérification de l'extraction des villes
print("\nAdresses avec ville extraite :")
print(df_adresses[['AdresseComplete', 'Ville']].head(20))  # Affiche les 20 premières lignes

# Fonction pour retirer la ville de l'adresse et obtenir uniquement l'adresse restante
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)

# Séparer les adresses avec et sans ville
df_avec_ville = df_adresses.dropna(subset=['Ville'])  # Garde celles avec une ville
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville'])  # Garde les adresses sans ville

# Sauvegarder le fichier final
df_adresses[['Adresse', 'Ville']].to_csv("adresses_nettoyees.csv", index=False)

# Affichage des résultats
print("\n📍 Adresses avec ville détectée :")
print(df_avec_ville[['Adresse', 'Ville']], "\n")

print("⚠️ Adresses sans ville détectée :")
print(df_sans_ville)


Liste des villes du catalogue :
['ABADOU', 'ABETEH', 'AFFELA IRHIR', 'AFOURER', 'AGADIR', 'AGDEZ', 'AGHBALA', 'AGHBALOU NKERDOUSS', 'AGHWINIT', 'AGOURAI']
Adresse normalisée : DR EL FIDE MEKKES MOULAY YACOUB
Correspondance exacte trouvée : MOULAY YACOUB
Adresse normalisée : COOP MASTOURA SBAA ROUADI MY YACOUB
Correspondance trouvée via similarité : MOULAY YACOUB
Adresse normalisée : 4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN FES
Correspondance exacte trouvée : FES
Adresse normalisée : DOUAR OLD BEN ABDELLAH OUDAYA FES BLIE
Correspondance exacte trouvée : FES
Adresse normalisée : AVE ALLAL BEN ABDELLAH 7 RUE AGHBALOU N
Correspondance trouvée via similarité : AGHBALOU NKERDOUSS
Adresse normalisée : Nø 58 AV IBN EL KHATIB / FES AGHBALOU N
Correspondance exacte trouvée : FES
Adresse normalisée : 40 RUE TOULOUSE CIL CASA 2
Correspondance trouvée via similarité : ASSA

Adresses avec ville extraite :
                                     AdresseComplete               Ville
0             

In [ ]:
import spacy
from fuzzywuzzy import process  # Utilisation de fuzzywuzzy pour un matching plus flexible
import pandas as pd
import re

# Charger le modèle spaCy français
nlp = spacy.load("fr_core_news_sm")

# Charger les fichiers CSV
df_adresses = pd.read_csv("Adresse1.csv")  # Contient les adresses complètes
df_villes = pd.read_csv("villes.csv")  # Contient la liste des villes

# Nettoyer le dataset des villes (enlever les NaN et mettre en majuscules pour homogénéiser)
df_villes = df_villes.dropna(subset=['NomVille'])  # Supprimer les lignes où NomVille est NaN
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()  # Liste des villes en majuscules

# Vérifier si les villes sont correctement chargées
print("Liste des villes du catalogue :")
print(catalogue_villes[:10])  # Affiche les 10 premières villes pour vérification

# Fonction pour extraire une ville similaire avec un seuil de similarité
def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None

    # Tokenisation et normalisation avec spaCy
    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])

    # Nettoyer les espaces et caractères spéciaux
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    print(f"Adresse normalisée : {normalized_input}")  # Débogage : affiche l'adresse normalisée

    # 1. Vérification directe de la correspondance exacte (avant get_close_matches)
    for ville in villes:
        if ville in normalized_input.upper():
            print(f"Correspondance exacte trouvée : {ville}")  # Débogage : correspondance exacte
            return ville.title()

    # 2. Chercher les villes similaires dans le catalogue en fonction de la similarité
    matches = process.extractOne(normalized_input.upper(), villes)

    # Débogage : vérifier si des correspondances sont trouvées
    if matches:
        best_match, score = matches
        if score >= seuil_similarite * 100:  # Seuil ajusté en pourcentage
            print(f"Correspondance trouvée via similarité : {best_match}")  # Débogage : ville similaire trouvée
            return best_match.title()
        else:
            print(f"Aucune correspondance suffisante trouvée, score : {score}")  # Débogage : si la correspondance n'est pas assez proche
    return None  # Retourne None si aucune ville similaire trouvée

# Fonction pour compléter les noms de villes incomplets
def completer_ville_incomplete(ville_incomplete, villes, seuil_similarite=80):
    """Complète le nom de la ville si un préfixe est fourni mais incomplet."""
    if ville_incomplete:
        # Utilisation de fuzzywuzzy pour trouver la meilleure correspondance
        best_match, score = process.extractOne(ville_incomplete.upper(), villes)
        if score >= seuil_similarite:  # Si le score est supérieur ou égal à 80 %
            return best_match.title()  # Retourne la ville complète trouvée
        else:
            print(f"Ville incomplète ignorée : {ville_incomplete}, score : {score}")  # Débogage : ville ignorée si le score est trop bas
    return None  # Retourne None si aucune correspondance n'est trouvée ou si le score est trop faible

# Appliquer la fonction pour extraire la ville similaire
df_adresses['Ville'] = df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes))

# Compléter les villes incomplètes en parcourant les adresses
# Si la similarité est inférieure à 80%, laisser la ville vide
df_adresses['Ville'] = df_adresses['Ville'].apply(lambda ville: completer_ville_incomplete(ville, catalogue_villes) if ville else None)

# Vérification de l'extraction des villes
print("\nAdresses avec ville extraite :")
print(df_adresses[['AdresseComplete', 'Ville']].head(20))  # Affiche les 20 premières lignes

# Fonction pour retirer la ville de l'adresse et obtenir uniquement l'adresse restante
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)

# Séparer les adresses avec et sans ville
df_avec_ville = df_adresses.dropna(subset=['Ville'])  # Garde celles avec une ville
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville'])  # Garde les adresses sans ville

# Sauvegarder le fichier final
df_adresses[['Adresse', 'Ville']].to_csv("adresses_nettoyees.csv", index=False)

# Affichage des résultats
print("\n📍 Adresses avec ville détectée :")
print(df_avec_ville[['Adresse', 'Ville']], "\n")

print("⚠️ Adresses sans ville détectée :")
print(df_sans_ville)


Liste des villes du catalogue :
['ABADOU', 'ABETEH', 'AFFELA IRHIR', 'AFOURER', 'AGADIR', 'AGDEZ', 'AGHBALA', 'AGHBALOU NKERDOUSS', 'AGHWINIT', 'AGOURAI']
Adresse normalisée : DR EL FIDE MEKKES MOULAY YACOUB
Correspondance exacte trouvée : MOULAY YACOUB
Adresse normalisée : COOP MASTOURA SBAA ROUADI MY YACOUB
Correspondance trouvée via similarité : MOULAY YACOUB
Adresse normalisée : 4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN FES
Correspondance exacte trouvée : FES
Adresse normalisée : DOUAR OLD BEN ABDELLAH OUDAYA FES BLIE
Correspondance exacte trouvée : FES
Adresse normalisée : AVE ALLAL BEN ABDELLAH 7 RUE AGHBALOU N
Correspondance trouvée via similarité : AGHBALOU NKERDOUSS
Adresse normalisée : Nø 58 AV IBN EL KHATIB / FES AGHBALOU N
Correspondance exacte trouvée : FES
Adresse normalisée : 40 RUE TOULOUSE CIL CASA 2
Correspondance trouvée via similarité : ASSA

Adresses avec ville extraite :
                                     AdresseComplete               Ville
0             

In [ ]:
import spacy
from fuzzywuzzy import process
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Charger le modèle spaCy français
nlp = spacy.load("fr_core_news_sm")

# Charger les fichiers CSV
df_adresses = pd.read_csv("Adresse1.csv")  # Contient les adresses complètes
df_villes = pd.read_csv("villes.csv")  # Contient la liste des villes

# Nettoyer le dataset des villes
df_villes = df_villes.dropna(subset=['NomVille'])
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()

# Liste de mots-clés pour les villes
mots_cles_villes = ['CASA', 'RABAT', 'MAROC', 'TANGER', 'MARRAKECH', 'FES']

# Vérifier si les villes sont correctement chargées
print("Liste des villes du catalogue :")
print(catalogue_villes[:10])

# Fonction pour obtenir l'embedding d'une adresse avec spaCy
def get_embedding(adresse):
    if not isinstance(adresse, str):
        return np.zeros(nlp.vocab.vectors.shape[1])  # Vecteur zéro si pas de texte
    doc = nlp(adresse)
    return doc.vector  # Vecteur moyen des tokens

# Fonction pour extraire une ville similaire avec un seuil de similarité et confiance
def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None, 0

    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    print(f"Adresse normalisée : {normalized_input}")

    # Vérification des mots-clés
    for mot_cle in mots_cles_villes:
        if mot_cle in normalized_input.upper():
            for ville in villes:
                if mot_cle in ville:
                    print(f"Correspondance par mot-clé : {ville}")
                    return ville.title(), 95

    # Correspondance exacte
    for ville in villes:
        if ville in normalized_input.upper():
            print(f"Correspondance exacte trouvée : {ville}")
            return ville.title(), 100

    # Similarité avec fuzzywuzzy
    matches = process.extractOne(normalized_input.upper(), villes)
    if matches:
        best_match, score = matches
        if score >= seuil_similarite * 100:
            print(f"Correspondance via similarité : {best_match}, score : {score}")
            return best_match.title(), score
    return None, 0

# Préparer les données d'entraînement pour l'auto-apprentissage
def preparer_donnees_entrainement(df):
    # Filtrer les lignes où une ville a été détectée avec confiance
    df_train = df.dropna(subset=['Ville']).copy()
    if len(df_train) == 0:
        print("Aucune donnée d'entraînement disponible.")
        return None, None, None

    # Générer les embeddings pour les adresses
    X = np.array([get_embedding(adresse) for adresse in df_train['AdresseComplete']])
    # Encoder les villes en labels numériques
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df_train['Ville'])

    return X, y, label_encoder

# Entraîner un modèle de classification
def entrainer_modele(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    modele = LogisticRegression(max_iter=1000)
    modele.fit(X_train, y_train)
    score = modele.score(X_test, y_test)
    print(f"Précision du modèle sur les données de test : {score:.2f}")
    return modele

# Prédire les villes manquantes avec le modèle
def predire_villes_manquantes(df, modele, label_encoder):
    df_sans_ville = df[df['Ville'].isna()].copy()
    if len(df_sans_ville) == 0 or modele is None:
        return df

    X_sans_ville = np.array([get_embedding(adresse) for adresse in df_sans_ville['AdresseComplete']])
    if X_sans_ville.shape[0] > 0:
        predictions = modele.predict(X_sans_ville)
        probas = modele.predict_proba(X_sans_ville)
        max_probas = probas.max(axis=1) * 100  # Pourcentage de confiance

        df_sans_ville['Ville'] = label_encoder.inverse_transform(predictions)
        df_sans_ville['Confiance'] = max_probas

        # Combiner les résultats
        df_final = pd.concat([df.dropna(subset=['Ville']), df_sans_ville])
        return df_final
    return df

# Étape 1 : Extraction initiale des villes avec confiance
df_adresses[['Ville', 'Confiance']] = pd.DataFrame(
    df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes)).tolist(),
    index=df_adresses.index
)

# Étape 2 : Préparer et entraîner le modèle
X, y, label_encoder = preparer_donnees_entrainement(df_adresses)
if X is not None and y is not None:
    modele = entrainer_modele(X, y)
    # Étape 3 : Prédire les villes manquantes
    df_adresses = predire_villes_manquantes(df_adresses, modele, label_encoder)
else:
    print("Pas de modèle entraîné, poursuite avec les résultats initiaux.")

# Vérification
print("\nAdresses avec ville extraite ou prédite :")
print(df_adresses[['AdresseComplete', 'Ville', 'Confiance']].head(20))

# Nettoyer l'adresse
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)

# Séparer les adresses avec et sans ville
df_avec_ville = df_adresses.dropna(subset=['Ville'])
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville', 'Confiance'])

# Sauvegarder le fichier final
df_adresses[['Adresse', 'Ville', 'Confiance']].to_csv("adresses_nettoyees.csv", index=False)

# Affichage des résultats
print("\n📍 Adresses avec ville détectée ou prédite :")
print(df_avec_ville[['Adresse', 'Ville', 'Confiance']], "\n")

print("⚠️ Adresses sans ville détectée :")
print(df_sans_ville)

Liste des villes du catalogue :
['ABADOU', 'ABETEH', 'AFFELA IRHIR', 'AFOURER', 'AGADIR', 'AGDEZ', 'AGHBALA', 'AGHBALOU NKERDOUSS', 'AGHWINIT', 'AGOURAI']
Adresse normalisée : DR EL FIDE MEKKES MOULAY YACOUB
Correspondance exacte trouvée : MOULAY YACOUB
Adresse normalisée : COOP MASTOURA SBAA ROUADI MY YACOUB
Correspondance via similarité : MOULAY YACOUB, score : 86
Adresse normalisée : 4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN FES
Correspondance par mot-clé : FES
Adresse normalisée : DOUAR OLD BEN ABDELLAH OUDAYA FES BLIE
Correspondance par mot-clé : FES
Adresse normalisée : AVE ALLAL BEN ABDELLAH 7 RUE AGHBALOU N
Correspondance via similarité : AGHBALOU NKERDOUSS, score : 86
Adresse normalisée : Nø 58 AV IBN EL KHATIB / FES AGHBALOU N
Correspondance par mot-clé : FES
Adresse normalisée : 40 RUE TOULOUSE CIL CASA 2
Correspondance par mot-clé : CASABLANCA
Précision du modèle sur les données de test : 0.00

Adresses avec ville extraite ou prédite :
                               

In [ ]:
import spacy
from fuzzywuzzy import process
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder


nlp = spacy.load("fr_core_news_sm")


df_adresses = pd.read_csv("Adresse1.csv")
df_villes = pd.read_csv("villes.csv")


df_villes = df_villes.dropna(subset=['NomVille'])
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()


mots_cles_villes = ['CASA', 'RABAT', 'MAROC', 'TANGER', 'MARRAKECH', 'FES']

print("Liste des villes du catalogue :")
print(catalogue_villes[:10])


def get_embedding(adresse):
    if not isinstance(adresse, str):
        return np.zeros(nlp.vocab.vectors.shape[1])  # Vecteur zéro si pas de texte
    doc = nlp(adresse)
    return doc.vector  # Vecteur moyen des tokens

# Fonction pour extraire une ville similaire avec un seuil de similarité et confiance
def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None, 0

    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    print(f"Adresse normalisée : {normalized_input}")

    # Vérification des mots-clés
    for mot_cle in mots_cles_villes:
        if mot_cle in normalized_input.upper():
            for ville in villes:
                if mot_cle in ville:
                    print(f"Correspondance par mot-clé : {ville}")
                    return ville.title(), 95

    # Correspondance exacte
    for ville in villes:
        if ville in normalized_input.upper():
            print(f"Correspondance exacte trouvée : {ville}")
            return ville.title(), 100

    # Similarité avec fuzzywuzzy
    matches = process.extractOne(normalized_input.upper(), villes)
    if matches:
        best_match, score = matches
        if score >= seuil_similarite * 100:
            print(f"Correspondance via similarité : {best_match}, score : {score}")
            return best_match.title(), score
    return None, 0

# Préparer les données d'entraînement pour l'auto-apprentissage
def preparer_donnees_entrainement(df):
    # Filtrer les lignes où une ville a été détectée avec confiance
    df_train = df.dropna(subset=['Ville']).copy()
    if len(df_train) == 0:
        print("Aucune donnée d'entraînement disponible.")
        return None, None, None

    # Générer les embeddings pour les adresses
    X = np.array([get_embedding(adresse) for adresse in df_train['AdresseComplete']])
    # Encoder les villes en labels numériques
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df_train['Ville'])

    return X, y, label_encoder

# Entraîner un modèle de classification
def entrainer_modele(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    modele = LogisticRegression(max_iter=1000)
    modele.fit(X_train, y_train)
    score = modele.score(X_test, y_test)
    print(f"Précision du modèle sur les données de test : {score:.2f}")
    return modele

# Prédire les villes manquantes avec le modèle
def predire_villes_manquantes(df, modele, label_encoder):
    df_sans_ville = df[df['Ville'].isna()].copy()
    if len(df_sans_ville) == 0 or modele is None:
        return df

    X_sans_ville = np.array([get_embedding(adresse) for adresse in df_sans_ville['AdresseComplete']])
    if X_sans_ville.shape[0] > 0:
        predictions = modele.predict(X_sans_ville)
        probas = modele.predict_proba(X_sans_ville)
        max_probas = probas.max(axis=1) * 100  # Pourcentage de confiance

        df_sans_ville['Ville'] = label_encoder.inverse_transform(predictions)
        df_sans_ville['Confiance'] = max_probas

        # Combiner les résultats
        df_final = pd.concat([df.dropna(subset=['Ville']), df_sans_ville])
        return df_final
    return df

# Étape 1 : Extraction initiale des villes avec confiance
df_adresses[['Ville', 'Confiance']] = pd.DataFrame(
    df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes)).tolist(),
    index=df_adresses.index
)

# Étape 2 : Préparer et entraîner le modèle
X, y, label_encoder = preparer_donnees_entrainement(df_adresses)
if X is not None and y is not None:
    modele = entrainer_modele(X, y)
    # Étape 3 : Prédire les villes manquantes
    df_adresses = predire_villes_manquantes(df_adresses, modele, label_encoder)
else:
    print("Pas de modèle entraîné, poursuite avec les résultats initiaux.")

# Vérification
print("\nAdresses avec ville extraite ou prédite :")
print(df_adresses[['AdresseComplete', 'Ville', 'Confiance']].head(20))

# Nettoyer l'adresse
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)

# Séparer les adresses avec et sans ville
df_avec_ville = df_adresses.dropna(subset=['Ville'])
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville', 'Confiance'])

# Sauvegarder le fichier final
df_adresses[['Adresse', 'Ville', 'Confiance']].to_csv("adresses_nettoyees.csv", index=False)

# Affichage des résultats
print("\n📍 Adresses avec ville détectée ou prédite :")
print(df_avec_ville[['Adresse', 'Ville', 'Confiance']], "\n")

print("⚠️ Adresses sans ville détectée :")
print(df_sans_ville)

Liste des villes du catalogue :
['ABADOU', 'ABETEH', 'AFFELA IRHIR', 'AFOURER', 'AGADIR', 'AGDEZ', 'AGHBALA', 'AGHBALOU NKERDOUSS', 'AGHWINIT', 'AGOURAI']
Adresse normalisée : DR EL FIDE MEKKES MOULAY YACOUB
Correspondance exacte trouvée : MOULAY YACOUB
Adresse normalisée : COOP MASTOURA SBAA ROUADI MY YACOUB
Correspondance via similarité : MOULAY YACOUB, score : 86
Adresse normalisée : 4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN FES
Correspondance par mot-clé : FES
Adresse normalisée : DOUAR OLD BEN ABDELLAH OUDAYA FES BLIE
Correspondance par mot-clé : FES
Adresse normalisée : AVE ALLAL BEN ABDELLAH 7 RUE AGHBALOU N
Correspondance via similarité : AGHBALOU NKERDOUSS, score : 86
Adresse normalisée : Nø 58 AV IBN EL KHATIB / FES AGHBALOU N
Correspondance par mot-clé : FES
Adresse normalisée : 40 RUE TOULOUSE CIL CASA 2
Correspondance par mot-clé : CASABLANCA
Précision du modèle sur les données de test : 0.00

Adresses avec ville extraite ou prédite :
                               

In [ ]:
import spacy
from fuzzywuzzy import process
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Charger le modèle spaCy français
nlp = spacy.load("fr_core_news_sm")

# Charger les fichiers CSV
df_adresses = pd.read_csv("Adresse1.csv")  # Contient les adresses complètes
df_villes = pd.read_csv("villes.csv")  # Contient la liste des villes

# Ajouter la colonne "AdresseAvantNettoyage"
df_adresses['Ancienne_Adresse'] = df_adresses['AdresseComplete']  # Sauvegarder l'adresse initiale avant le nettoyage

# Nettoyer le dataset des villes
df_villes = df_villes.dropna(subset=['NomVille'])
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()

# Liste de mots-clés pour les villes
mots_cles_villes = ['CASA', 'RABAT', 'MAROC', 'TANGER', 'MARRAKECH', 'FES']

# Vérifier si les villes sont correctement chargées
print("Liste des villes du catalogue :")
print(catalogue_villes[:10])

# Fonction pour obtenir l'embedding d'une adresse avec spaCy
def get_embedding(adresse):
    if not isinstance(adresse, str):
        return np.zeros(nlp.vocab.vectors.shape[1])  # Vecteur zéro si pas de texte
    doc = nlp(adresse)
    return doc.vector  # Vecteur moyen des tokens

# Fonction pour extraire une ville similaire avec un seuil de similarité et confiance
def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None, 0

    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    print(f"Adresse normalisée : {normalized_input}")

    # Vérification des mots-clés
    for mot_cle in mots_cles_villes:
        if mot_cle in normalized_input.upper():
            for ville in villes:
                if mot_cle in ville:
                    print(f"Correspondance par mot-clé : {ville}")
                    return ville.title(), 95

    # Correspondance exacte
    for ville in villes:
        if ville in normalized_input.upper():
            print(f"Correspondance exacte trouvée : {ville}")
            return ville.title(), 100

    # Similarité avec fuzzywuzzy
    matches = process.extractOne(normalized_input.upper(), villes)
    if matches:
        best_match, score = matches
        if score >= seuil_similarite * 100:
            print(f"Correspondance via similarité : {best_match}, score : {score}")
            return best_match.title(), score
    return None, 0

# Préparer les données d'entraînement pour l'auto-apprentissage
def preparer_donnees_entrainement(df):
    # Filtrer les lignes où une ville a été détectée avec confiance
    df_train = df.dropna(subset=['Ville']).copy()
    if len(df_train) == 0:
        print("Aucune donnée d'entraînement disponible.")
        return None, None, None

    # Générer les embeddings pour les adresses
    X = np.array([get_embedding(adresse) for adresse in df_train['AdresseComplete']])
    # Encoder les villes en labels numériques
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df_train['Ville'])

    return X, y, label_encoder

# Entraîner un modèle de classification
def entrainer_modele(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    modele = LogisticRegression(max_iter=1000)
    modele.fit(X_train, y_train)
    score = modele.score(X_test, y_test)
    print(f"Précision du modèle sur les données de test : {score:.2f}")
    return modele

# Prédire les villes manquantes avec le modèle
def predire_villes_manquantes(df, modele, label_encoder):
    df_sans_ville = df[df['Ville'].isna()].copy()
    if len(df_sans_ville) == 0 or modele is None:
        return df

    X_sans_ville = np.array([get_embedding(adresse) for adresse in df_sans_ville['AdresseComplete']])
    if X_sans_ville.shape[0] > 0:
        predictions = modele.predict(X_sans_ville)
        probas = modele.predict_proba(X_sans_ville)
        max_probas = probas.max(axis=1) * 100  # Pourcentage de confiance

        df_sans_ville['Ville'] = label_encoder.inverse_transform(predictions)
        df_sans_ville['Confiance'] = max_probas

        # Combiner les résultats
        df_final = pd.concat([df.dropna(subset=['Ville']), df_sans_ville])
        return df_final
    return df

# Étape 1 : Extraction initiale des villes avec confiance
df_adresses[['Ville', 'Confiance']] = pd.DataFrame(
    df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes)).tolist(),
    index=df_adresses.index
)

# Étape 2 : Préparer et entraîner le modèle
X, y, label_encoder = preparer_donnees_entrainement(df_adresses)
if X is not None and y is not None:
    modele = entrainer_modele(X, y)
    # Étape 3 : Prédire les villes manquantes
    df_adresses = predire_villes_manquantes(df_adresses, modele, label_encoder)
else:
    print("Pas de modèle entraîné, poursuite avec les résultats initiaux.")

# Vérification
print("\nAdresses avec ville extraite ou prédite :")
print(df_adresses[['Ancienne_Adresse', 'AdresseComplete', 'Ville', 'Confiance']].head(20))

# Nettoyer l'adresse
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)

# Séparer les adresses avec et sans ville
df_avec_ville = df_adresses.dropna(subset=['Ville'])
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville', 'Confiance'])

# Sauvegarder le fichier final
df_adresses[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']].to_csv("adresses_nettoyees.csv", index=False)

# Affichage des résultats
print("\n📍 Adresses avec ville détectée ou prédite :")
print(df_avec_ville[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']], "\n")

print("⚠️ Adresses sans ville détectée :")
print(df_sans_ville)


Liste des villes du catalogue :
['ABADOU', 'ABETEH', 'AFFELA IRHIR', 'AFOURER', 'AGADIR', 'AGDEZ', 'AGHBALA', 'AGHBALOU NKERDOUSS', 'AGHWINIT', 'AGOURAI']
Adresse normalisée : DR EL FIDE MEKKES MOULAY YACOUB
Correspondance exacte trouvée : MOULAY YACOUB
Adresse normalisée : COOP MASTOURA SBAA ROUADI MY YACOUB
Correspondance via similarité : MOULAY YACOUB, score : 86
Adresse normalisée : 4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN FES
Correspondance par mot-clé : FES
Adresse normalisée : DOUAR OLD BEN ABDELLAH OUDAYA FES BLIE
Correspondance par mot-clé : FES
Adresse normalisée : AVE ALLAL BEN ABDELLAH 7 RUE AGHBALOU N
Correspondance via similarité : AGHBALOU NKERDOUSS, score : 86
Adresse normalisée : Nø 58 AV IBN EL KHATIB / FES AGHBALOU N
Correspondance par mot-clé : FES
Adresse normalisée : 40 RUE TOULOUSE CIL CASA 2
Correspondance par mot-clé : CASABLANCA
Précision du modèle sur les données de test : 0.00

Adresses avec ville extraite ou prédite :
                               

In [ ]:
pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 11.6 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [ ]:
import spacy
import pandas as pd
import numpy as np
import re
from fuzzywuzzy import process
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score

# Charger le modèle spaCy français
nlp = spacy.load("fr_core_news_sm")

# Charger les fichiers CSV
df_adresses = pd.read_csv("Adresse1.csv")  # Contient les adresses complètes
df_villes = pd.read_csv("villes.csv")  # Contient la liste des villes

# Ajouter la colonne "AdresseAvantNettoyage"
df_adresses['Ancienne_Adresse'] = df_adresses['AdresseComplete']  # Sauvegarder l'adresse initiale avant le nettoyage

# Nettoyer le dataset des villes
df_villes = df_villes.dropna(subset=['NomVille'])
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()

# Liste de mots-clés pour les villes
mots_cles_villes = ['CASA', 'RABAT', 'MAROC', 'TANGER', 'MARRAKECH', 'FES']

# Fonction pour obtenir l'embedding d'une adresse avec spaCy
def get_embedding(adresse):
    if not isinstance(adresse, str):
        return np.zeros(nlp.vocab.vectors.shape[1])  # Vecteur zéro si pas de texte
    doc = nlp(adresse)
    return doc.vector  # Vecteur moyen des tokens

# Fonction pour extraire une ville similaire avec un seuil de similarité et confiance
def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None, 0

    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    print(f"Adresse normalisée : {normalized_input}")

    # Vérification des mots-clés
    for mot_cle in mots_cles_villes:
        if mot_cle in normalized_input.upper():
            for ville in villes:
                if mot_cle in ville:
                    print(f"Correspondance par mot-clé : {ville}")
                    return ville.title(), 95

    # Correspondance exacte
    for ville in villes:
        if ville in normalized_input.upper():
            print(f"Correspondance exacte trouvée : {ville}")
            return ville.title(), 100

    # Similarité avec fuzzywuzzy
    matches = process.extractOne(normalized_input.upper(), villes)
    if matches:
        best_match, score = matches
        if score >= seuil_similarite * 100:
            print(f"Correspondance via similarité : {best_match}, score : {score}")
            return best_match.title(), score
    return None, 0

# Préparer les données d'entraînement pour l'auto-apprentissage
def preparer_donnees_entrainement(df):
    # Filtrer les lignes où une ville a été détectée avec confiance
    df_train = df.dropna(subset=['Ville']).copy()
    if len(df_train) == 0:
        print("Aucune donnée d'entraînement disponible.")
        return None, None, None

    # Générer les embeddings pour les adresses
    X = np.array([get_embedding(adresse) for adresse in df_train['AdresseComplete']])
    # Encoder les villes en labels numériques
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df_train['Ville'])

    return X, y, label_encoder

# Créer et entraîner un modèle de Deep Learning
def entrainer_modele_deep_learning(X, y):
    model = Sequential()
    model.add(Dense(256, input_dim=X.shape[1], activation='relu'))  # Couche d'entrée avec activations ReLU
    model.add(Dropout(0.5))  # Dropout pour éviter le sur-apprentissage
    model.add(Dense(128, activation='relu'))  # Couche cachée
    model.add(Dense(64, activation='relu'))  # Couche cachée
    model.add(Dense(len(np.unique(y)), activation='softmax'))  # Couche de sortie (softmax pour la classification)

    # Compilation du modèle
    model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Entraîner le modèle
    model.fit(X, y, epochs=20, batch_size=32, validation_split=0.2)
    return model

# Prédire les villes manquantes avec le modèle
def predire_villes_manquantes(df, model, label_encoder):
    df_sans_ville = df[df['Ville'].isna()].copy()
    if len(df_sans_ville) == 0 or model is None:
        return df

    X_sans_ville = np.array([get_embedding(adresse) for adresse in df_sans_ville['AdresseComplete']])
    if X_sans_ville.shape[0] > 0:
        predictions = model.predict(X_sans_ville)
        predicted_labels = np.argmax(predictions, axis=1)
        df_sans_ville['Ville'] = label_encoder.inverse_transform(predicted_labels)
        df_sans_ville['Confiance'] = np.max(predictions, axis=1) * 100  # Probabilité maximale

        # Combiner les résultats
        df_final = pd.concat([df.dropna(subset=['Ville']), df_sans_ville])
        return df_final
    return df

# Étape 1 : Extraction initiale des villes avec confiance
df_adresses[['Ville', 'Confiance']] = pd.DataFrame(
    df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes)).tolist(),
    index=df_adresses.index
)

# Étape 2 : Préparer et entraîner le modèle
X, y, label_encoder = preparer_donnees_entrainement(df_adresses)
if X is not None and y is not None:
    model = entrainer_modele_deep_learning(X, y)
    # Étape 3 : Prédire les villes manquantes
    df_adresses = predire_villes_manquantes(df_adresses, model, label_encoder)
else:
    print("Pas de modèle entraîné, poursuite avec les résultats initiaux.")

# Vérification
print("\nAdresses avec ville extraite ou prédite :")
print(df_adresses[['Ancienne_Adresse', 'AdresseComplete', 'Ville', 'Confiance']].head(20))

# Nettoyer l'adresse
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)

# Séparer les adresses avec et sans ville
df_avec_ville = df_adresses.dropna(subset=['Ville'])
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville', 'Confiance'])

# Sauvegarder le fichier final
df_adresses[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']].to_csv("adresses_nettoyees.csv", index=False)

# Affichage des résultats
print("\n📍 Adresses avec ville détectée ou prédite :")
print(df_avec_ville[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']], "\n")

print("⚠️ Adresses sans ville détectée :")
print(df_sans_ville)


Adresse normalisée : DR EL FIDE MEKKES MOULAY YACOUB
Correspondance exacte trouvée : MOULAY YACOUB
Adresse normalisée : COOP MASTOURA SBAA ROUADI MY YACOUB
Correspondance via similarité : MOULAY YACOUB, score : 86
Adresse normalisée : 4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN FES
Correspondance par mot-clé : FES
Adresse normalisée : DOUAR OLD BEN ABDELLAH OUDAYA FES BLIE
Correspondance par mot-clé : FES
Adresse normalisée : HAY SAADA RUE 7 N 8 FES
Correspondance par mot-clé : FES
Adresse normalisée : IMMOUZER DU KANDAR PAR FES
Correspondance par mot-clé : FES
Adresse normalisée : 35 AVENUE DES FAR / FES
Correspondance par mot-clé : FES
Adresse normalisée : 209 RUE OKBA BEN NAFAA /SEFROU B / P 144 / SEFROU
Correspondance exacte trouvée : SEFROU
Adresse normalisée : AVE ALLAL BEN ABDELLAH 7 RUE AGHBALOU N
Correspondance via similarité : AGHBALOU NKERDOUSS, score : 86
Adresse normalisée : Nø 58 AV IBN EL KHATIB / FES AGHBALOU N
Correspondance par mot-clé : FES
Adresse normalisée : 

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.1250 - loss: 2.8185 - val_accuracy: 0.0000e+00 - val_loss: 2.7219
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.2500 - loss: 2.4095 - val_accuracy: 0.2857 - val_loss: 2.6629
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - accuracy: 0.6250 - loss: 1.8761 - val_accuracy: 0.2857 - val_loss: 2.6431
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.6667 - loss: 1.6855 - val_accuracy: 0.2857 - val_loss: 2.7058
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.6250 - loss: 1.4006 - val_accuracy: 0.2857 - val_loss: 2.8256
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.6250 - loss: 1.6599 - val_accuracy: 0.2857 - val_loss: 2.9313
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.6250 - loss: 1.5903 - val_accuracy: 0.2857 - val_loss: 3.0257
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.6250 - loss: 1.5677 - val_accuracy: 0.2857 - val_loss: 3.0196
Ep

In [ ]:
 import spacy
from fuzzywuzzy import process
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from pyspark.sql import SparkSession

# === Initialiser Spark ===
spark = SparkSession.builder \
    .appName("NormalisationAdresses") \
    .config("spark.jars", "C:\\spark1\\spark-3.5.5-bin-hadoop3\\jars\\mssql-jdbc-12.10.0.jre11.jar") \
    .getOrCreate()


# Configuration de la connexion source SQL Server
SQL_SERVER_CONFIG = {
    "host": "DESKTOP-5PJFL35",
    "port": "1433",
    "database": "MamdaAdresses",
    "user": "myuser",
    "password": "root"
}

jdbc_url = f"jdbc:sqlserver://{SQL_SERVER_CONFIG['host']}:{SQL_SERVER_CONFIG['port']};" \
           f"databaseName={SQL_SERVER_CONFIG['database']};encrypt=false;trustServerCertificate=true"




# === Chargement des données ===

df_adresses = spark.read.jdbc(url=jdbc_url, table="dbo.Adresse", properties=SQL_SERVER_CONFIG).toPandas()
df_villes = spark.read.jdbc(url=jdbc_url, table="dbo.villes", properties=SQL_SERVER_CONFIG).toPandas()

# === Plus------------------------------------------------

#df_adresses = df_adresses.repartition(130)
#df_villes = df_villes.repartition(130)


# === Préparation des données ===
df_adresses['Ancienne_Adresse'] = df_adresses['AdresseComplete']
df_villes = df_villes.dropna(subset=['NomVille'])
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()
catalogue_villes = df_villes['NomVille'].tolist()
mots_cles_villes = ['CASA', 'RABAT', 'MAROC', 'TANGER', 'MARRAKECH', 'FES']

# === Chargement du modèle spaCy ===
nlp = spacy.load("fr_core_news_sm")

def get_embedding(adresse):
    if not isinstance(adresse, str):
        return np.zeros(nlp.vocab.vectors.shape[1])
    return nlp(adresse).vector

def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    if not isinstance(adresse, str):
        return None, 0
    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()
    for mot_cle in mots_cles_villes:
        if mot_cle in normalized_input.upper():
            for ville in villes:
                if mot_cle in ville:
                    return ville.title(), 95
    for ville in villes:
        if ville in normalized_input.upper():
            return ville.title(), 100
    matches = process.extractOne(normalized_input.upper(), villes)
    if matches:
        best_match, score = matches
        if score >= seuil_similarite * 100:
            return best_match.title(), score
    return None, 0

df_adresses[['Ville', 'Confiance']] = pd.DataFrame(
    df_adresses['AdresseComplete'].apply(lambda x: extraire_ville_similaire(x, catalogue_villes)).tolist(),
    index=df_adresses.index
)

def preparer_donnees_entrainement(df):
    df_train = df.dropna(subset=['Ville']).copy()
    if len(df_train) == 0:
        return None, None, None
    X = np.array([get_embedding(adresse) for adresse in df_train['AdresseComplete']])
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df_train['Ville'])
    return X, y, label_encoder

def entrainer_modele(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    modele = LogisticRegression(max_iter=1000)
    modele.fit(X_train, y_train)
    score = modele.score(X_test, y_test)
    print(f"Précision du modèle : {score:.2f}")
    return modele

def predire_villes_manquantes(df, modele, label_encoder):
    df_sans_ville = df[df['Ville'].isna()].copy()
    if len(df_sans_ville) == 0 or modele is None:
        return df
    X_sans_ville = np.array([get_embedding(adresse) for adresse in df_sans_ville['AdresseComplete']])
    if X_sans_ville.shape[0] > 0:
        predictions = modele.predict(X_sans_ville)
        probas = modele.predict_proba(X_sans_ville)
        max_probas = probas.max(axis=1) * 100
        df_sans_ville['Ville'] = label_encoder.inverse_transform(predictions)
        df_sans_ville['Confiance'] = max_probas
        return pd.concat([df.dropna(subset=['Ville']), df_sans_ville])
    return df

X, y, label_encoder = preparer_donnees_entrainement(df_adresses)
if X is not None and y is not None:
    modele = entrainer_modele(X, y)
    df_adresses = predire_villes_manquantes(df_adresses, modele, label_encoder)

def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']), axis=1)
df_avec_ville = df_adresses.dropna(subset=['Ville'])
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville', 'Confiance'])

# === Sauvegarde dans SQL Server ===
df_resultats_spark = spark.createDataFrame(df_adresses[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']])

df_resultats_spark.write.jdbc(
    url=jdbc_url,
    table="dbo.Adresses_Nettoyees",
    mode="overwrite",
    properties=SQL_SERVER_CONFIG
)

# === Affichage final ===
print("\n📍 Résultat avec ville détectée/prévue :")
print(df_avec_ville[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']].head(10))
print("\n⚠️ Adresses sans ville détectée :")
print(df_sans_ville.head(10))

Précision du modèle : 0.60

📍 Résultat avec ville détectée/prévue :
                                    Ancienne_Adresse  \
0                                1 DERB EL HORRA FES   
1                      12 RUE WASSEL BNOU ATAE APPT2   
2          13 RUE ABI HAYANE TAOUHIDI    APPT 1 /FES   
3  15 AVENUE MLY RACHID IMM   FILALI APPT 4 ATLAS...   
4                     15 LOT MOUFARIH ALLEE CHRISTIA   
5                             18 RUE DU GABON /  FES   
6                       20 AVE IBN AICHA APPT 4  FES   
7   209 RUE OKBA BEN NAFAA /SEFROU  B/P 144 / SEFROU   
8                         21 RUE JORDANIE APPT 1/FES   
9  25 RUE ABI MOUSSA ACHARI LOTS  BENKIRANE FES A...   

                                             Adresse   Ville  Confiance  
0                                    1 DERB EL HORRA     Fes         95  
1                      12 RUE WASSEL BNOU ATAE APPT2    Assa         68  
2             13 RUE ABI HAYANE TAOUHIDI    APPT 1 /     Fes         95  
3     15 AVENUE MLY

In [ ]:
import spacy
from fuzzywuzzy import process
import pandas as pd
import re
import numpy as np
from pyspark.sql import SparkSession

# === Initialiser Spark ===
spark = SparkSession.builder \
    .appName("NormalisationAdresses") \
    .config("spark.jars", r"C:\spark1\spark-3.5.5-bin-hadoop3\jars\mssql-jdbc-12.10.0.jre11.jar") \
    .getOrCreate()

# Configuration de la connexion source SQL Server
SQL_SERVER_CONFIG = {
    "host": "DESKTOP-5PJFL35",
    "port": "1433",
    "database": "MamdaAdresses",
    "user": "myuser",
    "password": "root"
}

jdbc_url = (
    f"jdbc:sqlserver://{SQL_SERVER_CONFIG['host']}:{SQL_SERVER_CONFIG['port']};"
    f"databaseName={SQL_SERVER_CONFIG['database']};encrypt=false;trustServerCertificate=true"
)

# === Chargement des données ===
df_adresses = spark.read.jdbc(url=jdbc_url, table="dbo.Adresse", properties=SQL_SERVER_CONFIG).toPandas()
df_villes   = spark.read.jdbc(url=jdbc_url, table="dbo.villes",  properties=SQL_SERVER_CONFIG).toPandas()

# === Nettoyage et préparation ===
df_adresses['Ancienne_Adresse'] = df_adresses['AdresseComplete']

df_villes = df_villes.dropna(subset=['NomVille'])
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()

catalogue_villes = df_villes['NomVille'].tolist()
mots_cles_villes = ['CASA', 'RABAT', 'MAROC', 'TANGER', 'MARRAKECH', 'FES']

# === Chargement du modèle spaCy pour la tokenisation ===
nlp = spacy.load("fr_core_news_sm")

def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    """Tente d’extraire la ville d’une adresse par recherche floue et mots-clés."""
    if not isinstance(adresse, str):
        return None, 0

    doc = nlp(adresse)
    normalized_input = " ".join([token.text for token in doc])
    normalized_input = re.sub(r'\s+', ' ', normalized_input).strip()

    # Recherche par mots-clés
    for mot_cle in mots_cles_villes:
        if mot_cle in normalized_input.upper():
            for ville in villes:
                if mot_cle in ville:
                    return ville.title(), 95

    # Correspondance exacte
    for ville in villes:
        if ville in normalized_input.upper():
            return ville.title(), 100

    # Recherche floue
    match = process.extractOne(normalized_input.upper(), villes)
    if match:
        best_match, score = match
        if score >= seuil_similarite * 100:
            return best_match.title(), score

    return None, 0

# Application de l’algorithme d’extraction
df_adresses[['Ville', 'Confiance']] = pd.DataFrame(
    df_adresses['AdresseComplete']
    .apply(lambda x: extraire_ville_similaire(x, catalogue_villes))
    .tolist(),
    index=df_adresses.index
)

def nettoyer_adresse(adresse, ville):
    """Supprime le nom de la ville de l’adresse normalisée (si détecté)."""
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(
    lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']),
    axis=1
)

# Séparation pour inspection/qualité
df_avec_ville = df_adresses.dropna(subset=['Ville'])
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Confiance', 'Ville'])

# === Sauvegarde des résultats ===
df_resultats_spark = spark.createDataFrame(
    df_adresses[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']]
)

df_resultats_spark.write.jdbc(
    url=jdbc_url,
    table="dbo.Adresses_Nettoyees",
    mode="overwrite",
    properties=SQL_SERVER_CONFIG
)

# === Affichage rapide ===
print("\n📍 Adresses avec ville détectée :")
print(df_avec_ville[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']].head(10))

print("\n⚠️  Adresses sans ville détectée :")
print(df_sans_ville.head(10))



📍 Adresses avec ville détectée :
                                    Ancienne_Adresse  \
0                                1 DERB EL HORRA FES   
1                      12 RUE WASSEL BNOU ATAE APPT2   
2          13 RUE ABI HAYANE TAOUHIDI    APPT 1 /FES   
3  15 AVENUE MLY RACHID IMM   FILALI APPT 4 ATLAS...   
4                     15 LOT MOUFARIH ALLEE CHRISTIA   
5                             18 RUE DU GABON /  FES   
6                       20 AVE IBN AICHA APPT 4  FES   
7   209 RUE OKBA BEN NAFAA /SEFROU  B/P 144 / SEFROU   
8                         21 RUE JORDANIE APPT 1/FES   
9  25 RUE ABI MOUSSA ACHARI LOTS  BENKIRANE FES A...   

                                             Adresse   Ville  Confiance  
0                                    1 DERB EL HORRA     Fes         95  
1                      12 RUE WASSEL BNOU ATAE APPT2    Assa         68  
2             13 RUE ABI HAYANE TAOUHIDI    APPT 1 /     Fes         95  
3     15 AVENUE MLY RACHID IMM   FILALI APPT 4 ATLAS 

In [7]:
import spacy
from fuzzywuzzy import process
import pandas as pd
import re

# === Modèle spaCy ===
nlp = spacy.load("fr_core_news_sm")

# === Chargement des données ===
df_adresses = pd.read_csv("Adresse1.csv")
df_villes   = pd.read_csv("villes.csv")

# === Préparation ===
df_adresses['Ancienne_Adresse'] = df_adresses['AdresseComplete']
df_villes = df_villes.dropna(subset=['NomVille'])
df_villes['NomVille'] = df_villes['NomVille'].str.upper()

catalogue_villes = df_villes['NomVille'].tolist()
mots_cles_villes = ['CASA', 'RABAT', 'MAROC', 'TANGER', 'MARRAKECH', 'FES']

def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    """Retourne (ville, score) ou (None, 0)."""
    if not isinstance(adresse, str):
        return None, 0

    normalized = re.sub(r'\s+', ' ', " ".join([t.text for t in nlp(adresse)])).strip().upper()

    # 1) mots-clés
    for mot in mots_cles_villes:
        if mot in normalized:
            for ville in villes:
                if mot in ville:
                    return ville.title(), 95

    # 2) correspondance exacte
    for ville in villes:
        if ville in normalized:
            return ville.title(), 100

    # 3) fuzzy-matching
    match = process.extractOne(normalized, villes)
    if match:
        best, score = match
        if score >= seuil_similarite * 100:
            return best.title(), score

    return None, 0

# === Extraction des villes ===
df_adresses[['Ville', 'Confiance']] = pd.DataFrame(
    df_adresses['AdresseComplete']
    .apply(lambda x: extraire_ville_similaire(x, catalogue_villes))
    .tolist(),
    index=df_adresses.index
)

# === Nettoyage de l’adresse ===
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(
    lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']),
    axis=1
)

# === Séparation / export ===
df_avec_ville = df_adresses.dropna(subset=['Ville'])
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville', 'Confiance'])

df_adresses[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']].to_csv(
    "adresses_nettoyees.csv", index=False
)

print("\n📍 Adresses avec ville détectée :")
print(df_avec_ville[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']].head(20))

print("\n⚠️  Adresses sans ville détectée :")
print(df_sans_ville)



📍 Adresses avec ville détectée :
                                     Ancienne_Adresse  \
0                    DR EL FIDE MEKKES MOULAY YACOUB    
1                COOP MASTOURA SBAA ROUADI MY YACOUB    
2   4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 ...   
3             DOUAR OLD BEN ABDELLAH  OUDAYA FES BLIE   
4                          HAY SAADA RUE 7 N 8 FES      
5                       IMMOUZER DU KANDAR PAR FES      
6                        35 AVENUE DES  FAR  / FES      
7    209 RUE OKBA BEN NAFAA /SEFROU  B/P 144 / SEFROU   
8            AVE ALLAL BEN ABDELLAH 7 RUE  AGHBALOU N   
9            Nø 58 AV IBN EL KHATIB / FES  AGHBALOU N   
10                          18 RUE DU GABON /  FES      
11                              IMOUZZER DU KANDER      
12                         HAY BADR Nø 317 / FES .      
13  RAS EL MA SEJAA IN CHKEF  ZOUAGHAS MY YACOUB /FES   
14                      40 RUE TOULOUSE CIL CASA 2      
15                          RTE DE SEFROU KM 9 FES    

In [8]:
import spacy
from fuzzywuzzy import process
import pandas as pd
import re

# === Chargement du modèle spaCy ===
nlp = spacy.load("fr_core_news_sm")

# === Chargement des données depuis CSV ===
df_adresses = pd.read_csv("Adresse1.csv")
df_villes   = pd.read_csv("villes.csv")

# === Préparation des données ===
df_adresses['Ancienne_Adresse'] = df_adresses['AdresseComplete']

df_villes = df_villes.dropna(subset=['NomVille'])
df_villes['NomVille'] = df_villes['NomVille'].astype(str).str.upper()

catalogue_villes = df_villes['NomVille'].tolist()
mots_cles_villes = ['CASA', 'RABAT', 'MAROC', 'TANGER', 'MARRAKECH', 'FES']

def extraire_ville_similaire(adresse, villes, seuil_similarite=0.6):
    """Renvoie (ville, score) ou (None, 0)."""
    if not isinstance(adresse, str):
        return None, 0

    # Normalisation simple
    normalized = re.sub(r'\s+', ' ', " ".join([t.text for t in nlp(adresse)])).strip().upper()

    # 1) Par mot-clé
    for mot in mots_cles_villes:
        if mot in normalized:
            for ville in villes:
                if mot in ville:
                    return ville.title(), 95

    # 2) Correspondance exacte
    for ville in villes:
        if ville in normalized:
            return ville.title(), 100

    # 3) Fuzzy-matching
    match = process.extractOne(normalized, villes)
    if match:
        best, score = match
        if score >= seuil_similarite * 100:
            return best.title(), score

    return None, 0

# === Extraction des villes ===
df_adresses[['Ville', 'Confiance']] = pd.DataFrame(
    df_adresses['AdresseComplete']
    .apply(lambda x: extraire_ville_similaire(x, catalogue_villes))
    .tolist(),
    index=df_adresses.index
)

# === Nettoyage de l’adresse (on retire le nom de la ville) ===
def nettoyer_adresse(adresse, ville):
    if pd.notna(ville) and isinstance(adresse, str):
        adresse = re.sub(r'\b' + re.escape(ville) + r'\b', '', adresse, flags=re.IGNORECASE)
    return adresse.strip()

df_adresses['Adresse'] = df_adresses.apply(
    lambda row: nettoyer_adresse(row['AdresseComplete'], row['Ville']),
    axis=1
)

# === Séparation pour inspection facultative ===
df_avec_ville = df_adresses.dropna(subset=['Ville'])
df_sans_ville = df_adresses[df_adresses['Ville'].isna()].drop(columns=['Ville', 'Confiance'])

# === Sauvegarde du résultat ===
df_adresses[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']].to_csv(
    "adresses_nettoyees.csv", index=False
)

# === Affichage rapide ===
print("\n📍 Adresses avec ville détectée :")
print(df_avec_ville[['Ancienne_Adresse', 'Adresse', 'Ville', 'Confiance']].head(10))

print("\n⚠️  Adresses sans ville détectée :")
print(df_sans_ville.head(10))



📍 Adresses avec ville détectée :
                                    Ancienne_Adresse  \
0                   DR EL FIDE MEKKES MOULAY YACOUB    
1               COOP MASTOURA SBAA ROUADI MY YACOUB    
2  4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 ...   
3            DOUAR OLD BEN ABDELLAH  OUDAYA FES BLIE   
4                         HAY SAADA RUE 7 N 8 FES      
5                      IMMOUZER DU KANDAR PAR FES      
6                       35 AVENUE DES  FAR  / FES      
7   209 RUE OKBA BEN NAFAA /SEFROU  B/P 144 / SEFROU   
8           AVE ALLAL BEN ABDELLAH 7 RUE  AGHBALOU N   
9           Nø 58 AV IBN EL KHATIB / FES  AGHBALOU N   

                                            Adresse               Ville  \
0                                 DR EL FIDE MEKKES       Moulay Yacoub   
1               COOP MASTOURA SBAA ROUADI MY YACOUB       Moulay Yacoub   
2  4 RUE MOHAMED EL GHAZI APPT 13 RCE REDA ETG 2 VN                 Fes   
3              DOUAR OLD BEN ABDELLAH  OUDAYA  BL